In [7]:
import numpy as np    
import pandas as pd   
from sklearn.preprocessing import LabelEncoder


In [3]:
df=pd.read_csv('cleaned_lalpurja_land_final_after_eda.csv')

In [4]:
df_ml_lp = df.copy()


In [6]:
# STEP 1 — LOG TRANSFORM
# Why: land_size_aana is heavily right skewed (skewness 3.098)
# Log transform normalizes it for better model performance
# ─────────────────────────────────────────
df_ml_lp['log_land'] = np.log1p(df_ml_lp['land_size_aana'])


In [8]:
df_ml_lp.columns

Index(['district', 'property_type', 'property_face', 'road_type',
       'municipality', 'ward_no', 'neighborhood', 'price_per_aana',
       'land_size_aana', 'road_width_feet', 'facing_road_width', 'hospital_m',
       'airport_m', 'pharmacy_m', 'bhatbhateni_m', 'school_m',
       'public_transport_m', 'police_station_m', 'ring_road_m',
       'ring_road_proximity', 'comm_road_premium', 'is_corner_plot',
       'log_land'],
      dtype='object')

In [10]:
# ─────────────────────────────────────────
# STEP 2 — LABEL ENCODE LOW CARDINALITY COLUMNS
# Why: ML models need numeric inputs not strings
# Label encoding assigns integer to each category
# district: 3 values, road_type: 2 values,
# property_face: 8 values, property_type: 3 values
# ─────────────────────────────────────────
le = LabelEncoder()
for col in ['district', 'road_type', 'property_type', 'property_face']:
    df_ml_lp[col] = le.fit_transform(df_ml_lp[col].astype(str))


In [11]:
# ─────────────────────────────────────────
# STEP 3 — TARGET ENCODE HIGH CARDINALITY COLUMNS
# Why: neighborhood (103 unique) and municipality (20 unique)
# are too many categories for label encoding
# Target encoding replaces each category with mean log price
# so the model gets price signal directly from location
# ─────────────────────────────────────────
for col in ['neighborhood', 'municipality']:
    target_map  = df_ml_lp.groupby(col)['price_per_aana'].apply(
        lambda x: np.log1p(x).mean()
    )
    global_mean = np.log1p(df_ml_lp['price_per_aana']).mean()
    df_ml_lp[col + '_encoded'] = df_ml_lp[col].map(target_map).fillna(global_mean)
    df_ml_lp.drop(columns=[col], inplace=True)

In [12]:
# ─────────────────────────────────────────
# STEP 5 — VERIFY
# ─────────────────────────────────────────
print("=== Object columns remaining ===")
obj = df_ml_lp.select_dtypes(include='object').columns.tolist()
print(obj if obj else "✅ None")

print("\n=== Nulls ===")
nulls = df_ml_lp.isnull().sum()
print(nulls[nulls > 0] if nulls[nulls > 0].any() else "✅ No nulls")

print(f"\n=== Final Shape: {df_ml_lp.shape} ===")
print(f"=== Columns: {list(df_ml_lp.columns)} ===")

=== Object columns remaining ===
✅ None

=== Nulls ===
✅ No nulls

=== Final Shape: (1214, 23) ===
=== Columns: ['district', 'property_type', 'property_face', 'road_type', 'ward_no', 'price_per_aana', 'land_size_aana', 'road_width_feet', 'facing_road_width', 'hospital_m', 'airport_m', 'pharmacy_m', 'bhatbhateni_m', 'school_m', 'public_transport_m', 'police_station_m', 'ring_road_m', 'ring_road_proximity', 'comm_road_premium', 'is_corner_plot', 'log_land', 'neighborhood_encoded', 'municipality_encoded'] ===


In [13]:
# ─────────────────────────────────────────
# ADVANCED FEATURE ENGINEERING 
# ─────────────────────────────────────────


In [14]:
# 1 — Urban Centrality Score
# How it works: Combines airport and ring road distance into one score
# The closer to both → lower denominator → higher score → higher expected price
# Example:
#   Plot A: airport_m=5000, ring_road_m=2000 → 1/(log(5001)+log(2001)) = 0.118
#   Plot B: airport_m=15000, ring_road_m=8000 → 1/(log(15001)+log(8001)) = 0.096
#   Plot A scores higher → correctly identified as more urban/central
df_ml_lp['urban_centrality'] = 1 / (np.log1p(df_ml_lp['airport_m']) +
                                     np.log1p(df_ml_lp['ring_road_m']))

In [15]:
# 2 — Amenity Access Score
# How it works: Sums reciprocal log of 4 key amenity distances
# More amenities nearby → higher score → higher expected price
# Example:
#   hospital_m=500, school_m=200, pharmacy_m=100, public_transport_m=50
#   score = 1/log(501) + 1/log(201) + 1/log(101) + 1/log(51)
#         = 0.162 + 0.187 + 0.217 + 0.257 = 0.823  ← high access
#   hospital_m=5000, school_m=3000, pharmacy_m=2000, public_transport_m=1000
#   score = 0.116 + 0.124 + 0.130 + 0.145 = 0.515  ← lower access
df_ml_lp['amenity_access_score'] = (
    1 / np.log1p(df_ml_lp['hospital_m']) +
    1 / np.log1p(df_ml_lp['school_m']) +
    1 / np.log1p(df_ml_lp['pharmacy_m']) +
    1 / np.log1p(df_ml_lp['public_transport_m'])
)

In [16]:
# 3 — Plot Value Score
# How it works: Multiplies log land size by log road width
# Larger plot + wider road = more valuable combination
# Example:
#   Plot A: land=8 aana, road=20ft → log(9) × log(21) = 2.197 × 3.045 = 6.69
#   Plot B: land=4 aana, road=13ft → log(5) × log(14) = 1.609 × 2.639 = 4.25
#   Plot A scores higher → correctly identified as more valuable plot
df_ml_lp['plot_value_score'] = (
    np.log1p(df_ml_lp['land_size_aana']) *
    np.log1p(df_ml_lp['road_width_feet'])
)

In [17]:
# 4 — Commercial Zone Score
# How it works: property_type (encoded 0/1/2) × road_width_feet
# Non-residential plots on wide roads get highest score
# Example:
#   Residential plot (type=0), road=20ft → 0 × 20 = 0 (no premium)
#   Commercial plot (type=1), road=20ft  → 1 × 20 = 20 (road premium applies)
#   Semi-comm plot (type=2), road=30ft  → 2 × 30 = 60 (highest premium)
df_ml_lp['commercial_zone_score'] = (
    df_ml_lp['property_type'] *
    df_ml_lp['road_width_feet']
)

In [18]:
# 5 — Neighborhood × District Interaction
# How it works: Multiplies neighborhood price signal by district code
# Same neighborhood name means different things in different districts
# Example:
#   "Imadol" in Lalitpur (district=1): 15.2 × 1 = 15.2
#   "Imadol" in Kathmandu (district=0): 15.2 × 0 = 0
#   Model learns district-specific neighborhood premiums
df_ml_lp['neighborhood_x_district'] = (
    df_ml_lp['neighborhood_encoded'] *
    df_ml_lp['district']
)

In [19]:
# 6 — Municipality × Ward Interaction
# How it works: Multiplies municipality price signal by ward number
# Ward 6 in Kathmandu ≠ Ward 6 in Bhaktapur
# Example:
#   Kathmandu municipality_encoded=15.8, ward=6 → 15.8 × 6 = 94.8
#   Bhaktapur municipality_encoded=14.9, ward=6 → 14.9 × 6 = 89.4
#   Model learns ward-level price differences within each municipality
df_ml_lp['municipality_x_ward'] = (
    df_ml_lp['municipality_encoded'] *
    df_ml_lp['ward_no']
)

In [20]:
# 7 — Road Access Quality Score
# How it works: Adds log of both road widths together
# A plot with wide facing road AND wide access road is most valuable
# Example:
#   Plot A: road_width=20ft, facing_road=35ft → log(21)+log(36) = 3.04+3.58 = 6.62
#   Plot B: road_width=13ft, facing_road=28ft → log(14)+log(29) = 2.64+3.37 = 6.01
#   Plot A scores higher → better road access on both sides
df_ml_lp['road_access_quality'] = (
    np.log1p(df_ml_lp['road_width_feet']) +
    np.log1p(df_ml_lp['facing_road_width'])
)

In [21]:
# ─────────────────────────────────────────
# VERIFY
# ─────────────────────────────────────────
print(f"Features before: 23")
print(f"Features after:  {df_ml_lp.shape[1]}")
print(f"\nNew features added:")
new_features = ['urban_centrality', 'amenity_access_score', 'plot_value_score',
                'commercial_zone_score', 'neighborhood_x_district',
                'municipality_x_ward', 'road_access_quality']
for f in new_features:
    print(f"  {f}: min={df_ml_lp[f].min():.3f}, max={df_ml_lp[f].max():.3f}")

print(f"\nNulls: {df_ml_lp.isnull().sum().sum()}")

Features before: 23
Features after:  30

New features added:
  urban_centrality: min=0.050, max=0.101
  amenity_access_score: min=0.477, max=1.017
  plot_value_score: min=1.662, max=13.720
  commercial_zone_score: min=0.000, max=90.000
  neighborhood_x_district: min=0.000, max=31.741
  municipality_x_ward: min=14.478, max=518.504
  road_access_quality: min=3.584, max=8.329

Nulls: 0


In [23]:
# ─────────────────────────────────────────
# SAVE FEATURE ENGINEERED LALPURJA DATASET
# This is the final version ready for modeling with:
# - 23 original features after encoding
# - 7 new engineered features
# - 30 total features, 1214 rows, zero nulls
# ─────────────────────────────────────────
df_ml_lp.to_csv('lalpurja_dataset_ready_after_feature_engineering.csv', index=False)

print(f"✅ Saved lalpurja_dataset_ready_after_feature_engineering.csv")
print(f"✅ Shape: {df_ml_lp.shape}")
print(f"✅ Nulls: {df_ml_lp.isnull().sum().sum()}")
print(f"✅ Columns: {list(df_ml_lp.columns)}")

✅ Saved lalpurja_dataset_ready_after_feature_engineering.csv
✅ Shape: (1214, 30)
✅ Nulls: 0
✅ Columns: ['district', 'property_type', 'property_face', 'road_type', 'ward_no', 'price_per_aana', 'land_size_aana', 'road_width_feet', 'facing_road_width', 'hospital_m', 'airport_m', 'pharmacy_m', 'bhatbhateni_m', 'school_m', 'public_transport_m', 'police_station_m', 'ring_road_m', 'ring_road_proximity', 'comm_road_premium', 'is_corner_plot', 'log_land', 'neighborhood_encoded', 'municipality_encoded', 'urban_centrality', 'amenity_access_score', 'plot_value_score', 'commercial_zone_score', 'neighborhood_x_district', 'municipality_x_ward', 'road_access_quality']
